In [ ]:
import wot
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scanpy as sc
from pathlib import Path
from collections import defaultdict
from scipy import sparse

import chronopack as cp


## Waddington‑OT (Transport maps)
Following the official [WOT notebooks](https://github.com/broadinstitute/wot/tree/master/notebooks), we compute transport maps between successive time points.
Make sure `adata.obs['time']` (integer or float) exists and reflects the sampling time in *hours/days* used in the Seurat notebook.

In [ ]:
adata_hvg = sc.read_h5ad('../output/adata_hvg.1e4-variant.chk0.h5ad')
adata_hvg.obs['time'] = pd.to_numeric(adata_hvg.obs['time'].astype(str))
adata_hvg.X = adata_hvg.layers['corrected'].copy()

TMAP_DIR = '../intmd/tmaps.SingleCell.1e4/'

wot_out = cp.tl.get_trajectories(
    adata_hvg,
    tmap_dir=TMAP_DIR,
    ot_model_kwargs=dict(epsilon=0.05, lambda1=1, lambda2=50, growth_iters=3, day_field='time'),
    compute_transport_maps=True,
    transition_topk=30,
    symmetrize='maximum',
    graph_key='ot_graph',
    umap_key='ot_umap',
    umap_random_state=10,
)

tmaps = wot_out['tmaps']
transitions_long = wot_out['transitions_long']
tmats = wot_out['tmats']
T_full = wot_out['T_full']
T_pruned = wot_out['T_pruned']
T_sym = wot_out['T_sym']
key = 'ot_umap'
key_out = key


# Construct forward transition kernel (+ pruned & symmetric variants)

# Cluster trajectories

In [ ]:
S = cp.tl.cluster_cells_to_trajectories(
    adata_hvg,
    T_pruned,
    cluster_key="traj_clusters",
    resolution=0.3,
    terminal_topk=200,
    graph_topk=500,
    random_state=0,
)
sc.pl.embedding(
    adata_hvg,
    basis="ot_umap",
    color=["time", "traj_clusters", "total_counts", "n_genes_by_counts", "NFKBIA", "HBE1"],
    ncols=3,
    cmap="viridis",
)


# Calculate terminal fates & trajectory cluster mass retention

In [ ]:
F, fate_names = cp.tl.compute_terminal_fate_probabilities(adata_hvg, terminal_cluster_key="traj_clusters", out_key="terminal_fate_probs_traj")
cp.pl.cell_trajectories.fate_probabilities(adata_hvg, embedding_basis=key_out, size=15)

In [ ]:
out = cp.pl.cell_trajectories.mass_retention(adata_hvg, S=T_pruned, 
                                    cluster_key="traj_clusters", 
                                    n_reps=50, clusters_to_plot=["0","1"], 
                                    embedding_basis="ot_umap", size=20)

In [ ]:
cp.pl.cell_trajectories.proportions_over_time(adata_hvg)

# Visualize on original UMAP

In [7]:
tmp = sc.read_h5ad('../output/adata_hvg.1e4-variant.chk0.h5ad')
adata_hvg.obsm['X_umap_original'] = tmp.obsm['X_umap'].copy()

sc.tl.leiden(tmp, resolution=0.3)
adata_hvg.obs['leiden'] = tmp.obs['leiden'].copy()

In [8]:
adata = sc.read_h5ad('../output/adata_full.1e4-variant.chk0.h5ad')

adata.obs = adata_hvg.obs.copy()
adata.obsm = adata_hvg.obsm.copy()
adata.obsp = adata_hvg.obsp.copy()
adata.uns = adata_hvg.uns.copy()

adata.write_h5ad('../output/adata_full.1e4-variant.chk1.h5ad')
adata_hvg.write_h5ad('../output/adata_hvg.1e4-variant.chk1.h5ad')